In [23]:
import polars as pl
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import joblib
from sklearn.preprocessing import MinMaxScaler


In [24]:
print("Phase 1: Loading, Aggregating, and Unifying Raw Data...")

# 1. Load Resource Data (Containers - 30s interval)
df_resource = pl.read_csv("Data/MSResource_0.csv")

df_resource = df_resource.with_columns((pl.col("timestamp") // 60000 * 60000).alias("timestamp"))
df_resource = df_resource.group_by(["timestamp", "msname", "msinstanceid", "nodeid"]).agg([
    pl.col("instance_cpu_usage").mean(),
    pl.col("instance_memory_usage").mean()
])

# 2. Load Node Data (Bare-Metal Hosts - 30s interval)
df_node = pl.read_csv("Data/Node_0.csv")
# FIX: Snap to 60s and average the 30s polling intervals BEFORE joining
df_node = df_node.with_columns((pl.col("timestamp") // 60000 * 60000).alias("timestamp"))
df_node = df_node.group_by(["timestamp", "nodeid"]).agg([
    pl.col("node_cpu_usage").mean(),
    pl.col("node_memory_usage").mean()
])

# 3. Load MSRTQps Data (Traffic & Performance - 60s interval)
df_qps = pl.read_csv("Data/MSRTQps_0.csv")
df_qps = df_qps.with_columns((pl.col("timestamp") // 60000 * 60000).alias("timestamp"))

# Pivot the QPS table
df_qps_pivoted = df_qps.pivot(
    values="value", 
    index=["timestamp", "msname", "msinstanceid"], 
    on="metric"
)

# 4. The Join: Merge the cleanly aggregated 60s tables
df_merged = df_resource.join(df_node, on=["timestamp", "nodeid"], how="left")
df_merged = df_merged.join(df_qps_pivoted, on=["timestamp", "msname", "msinstanceid"], how="left")

Phase 1: Loading, Aggregating, and Unifying Raw Data...


In [26]:
df_merged.head()
df_merged.shape

(5782203, 16)

In [28]:
print("Phase 2: Aggregating to Service Level...")

# 1. Ensure required traffic columns exist (creates them if no service used them in this chunk)
traffic_cols = ["providerRPC_MCR", "consumerMQ_MCR", "HTTP_MCR"]
for col in traffic_cols:
    if col not in df_merged.columns:
        df_merged = df_merged.with_columns(pl.lit(0.0).alias(col))

# 2. Ensure required RT columns exist
rt_cols = ["providerRPC_RT", "consumerMQ_RT", "HTTP_RT"]
for col in rt_cols:
    if col not in df_merged.columns:
        df_merged = df_merged.with_columns(pl.lit(None).cast(pl.Float64).alias(col))

# 3. Calculate Container-Level True Traffic and Avg RT
# THE FIX: We use .fill_null(0) to prevent the "Pivot Trap" from wiping out the math
df_merged = df_merged.with_columns([
    pl.sum_horizontal([pl.col(c).fill_null(0) for c in traffic_cols]).alias("container_traffic"),
    pl.mean_horizontal(rt_cols).alias("container_rt")
])

# 4. Aggregate up to the msname (Service) level
df_service = df_merged.group_by(["msname", "timestamp"]).agg([
    # Target Variables: Summing container usage gives us the total scale of the service
    pl.col("instance_cpu_usage").sum().alias("total_cpu_demand"),
    pl.col("instance_memory_usage").sum().alias("total_memory_demand"),
    
    # Workload: Summing all incoming calls across all containers
    pl.col("container_traffic").sum().alias("total_traffic"),
    
    # Environment & Interference: Averaging the physical host stress
    pl.col("node_cpu_usage").mean().alias("avg_node_stress"),
    
    # Performance: Averaging the response times
    pl.col("container_rt").mean().alias("avg_response_time"),
    
    # Meta: Keeping track of the number of containers
    pl.len().alias("replica_count")
]).sort(["msname", "timestamp"])

print(df_service.head())

Phase 2: Aggregating to Service Level...
shape: (5, 8)
┌────────────┬───────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┐
│ msname     ┆ timestamp ┆ total_cpu_ ┆ total_memo ┆ total_tra ┆ avg_node_ ┆ avg_respo ┆ replica_c │
│ ---        ┆ ---       ┆ demand     ┆ ry_demand  ┆ ffic      ┆ stress    ┆ nse_time  ┆ ount      │
│ str        ┆ i64       ┆ ---        ┆ ---        ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│            ┆           ┆ f64        ┆ f64        ┆ f64       ┆ f64       ┆ f64       ┆ u32       │
╞════════════╪═══════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 002251d412 ┆ 0         ┆ 69.644772  ┆ 298.817297 ┆ 46808.901 ┆ 0.702467  ┆ 37.767357 ┆ 437       │
│ 3496684687 ┆           ┆            ┆            ┆ 161       ┆           ┆           ┆           │
│ c2acad43bd ┆           ┆            ┆            ┆           ┆           ┆           ┆           │
│ …          ┆           ┆          

In [29]:
df_merged.head()
df_service.shape

(78180, 8)

In [ ]:
# ==========================================
# QUICK SAFETY FILTER (Bypassing Phase 3)
# ==========================================
valid_services = df_service.group_by("msname").agg(
    pl.col("total_traffic").sum().alias("lifetime_traffic")
).filter(pl.col("lifetime_traffic") > 0).select("msname")

df_final_1m = df_service.join(valid_services, on="msname", how="inner")

df_final_1m.shape
# # ---> NEW: Save the cleaned 1-minute dataset to a CSV file <---
# df_final_1m.write_csv("LSTM_Final_Cleaned_1m.csv")
# print("Successfully saved cleaned data to 'LSTM_Final_Cleaned_1m.csv'")

Successfully saved cleaned data to 'LSTM_Final_Cleaned_1m.csv'


In [ ]:
print("Phase 4: Scaling Features...")

# 1. Select the specific columns the LSTM will use
features = ["total_traffic", "avg_node_stress", "total_memory_demand", "avg_response_time", "total_cpu_demand"]
data_matrix = df_final_1m.select(features).to_numpy()

# 2. Compress the data into a [3] range
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data_matrix)

# 3. Save the scaler to your disk
# This is CRITICAL. When your LSTM outputs a prediction like "0.85", 
# you will need to load this exact scaler to decode it back into real CPU cores!
joblib.dump(scaler, "v2_1m_scaler.pkl")
print("Saved scaler as 'v2_1m_scaler.pkl'")

# 4. Attach the scaled data back to the microservice names and timestamps
df_scaled = df_final_1m.select(["msname", "timestamp"]).with_columns([
    pl.Series("scaled_traffic", data_scaled[:, 0]),
    pl.Series("scaled_node_stress", data_scaled[:, 1]),
    pl.Series("scaled_mem", data_scaled[:, 2]),
    pl.Series("scaled_rt", data_scaled[:, 3]),
    pl.Series("scaled_cpu", data_scaled[:, 4]) # This is your Target (y)
])

print("\nPhase 4 Complete! Scaled Data Preview:")
print(df_scaled.head())


Phase 4: Scaling Features...
Saved scaler as 'v2_1m_scaler.pkl'

Phase 4 Complete! Scaled Data Preview:
shape: (5, 7)
┌────────────────┬───────────┬───────────────┬───────────────┬────────────┬───────────┬────────────┐
│ msname         ┆ timestamp ┆ scaled_traffi ┆ scaled_node_s ┆ scaled_mem ┆ scaled_rt ┆ scaled_cpu │
│ ---            ┆ ---       ┆ c             ┆ tress         ┆ ---        ┆ ---       ┆ ---        │
│ str            ┆ i64       ┆ ---           ┆ ---           ┆ f64        ┆ f64       ┆ f64        │
│                ┆           ┆ f64           ┆ f64           ┆            ┆           ┆            │
╞════════════════╪═══════════╪═══════════════╪═══════════════╪════════════╪═══════════╪════════════╡
│ 002251d4123496 ┆ 0         ┆ 0.053902      ┆ 0.817038      ┆ 0.159503   ┆ 0.008676  ┆ 0.043204   │
│ 684687c2acad43 ┆           ┆               ┆               ┆            ┆           ┆            │
│ bd…            ┆           ┆               ┆               ┆            

In [32]:
df_scaled.shape

(77700, 7)